# GRetriever+PCST Server

Serves the trained GRetriever (GNN+MLP+LoRA MedGemma) as a FastAPI endpoint
for ICD→HPO disambiguation. Called from `01_llm_demo.ipynb` alongside the
GNN reranker and embedding APIs.

**Endpoints:**
- `GET /healthz` — model info and status
- `POST /disambiguate` — given ICD code + HPO candidates, return GRetriever's prediction
- `POST /compare` — run both text-only and GRetriever, return side-by-side

**Prerequisites:**
- `ontology_graph.pkl` in Google Drive
- `gretriever_pcst_checkpoint.pt` in Google Drive
- `gretriever_pcst_lora/` directory in Google Drive (LoRA adapter)
- ngrok domain configured in Colab secrets as `NGROK_DOMAIN_GRETRIEVER`

In [1]:
%%capture
!pip install -qq fastapi uvicorn pyngrok
!pip install -qq torch torch_geometric
!pip install -qq transformers accelerate bitsandbytes peft pcst_fast

In [2]:
from __future__ import annotations

import os
import re
import pickle
from collections import defaultdict
from pathlib import Path
from typing import List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

import pcst_fast
from pyngrok import ngrok
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from google.colab import userdata
import uvicorn

In [3]:
NGROK_TOKEN = userdata.get("NGROK_TOKEN")
NGROK_DOMAIN_GRETRIEVER = userdata.get("NGROK_DOMAIN_GRETRIEVER")

os.environ["NGROK_TOKEN"] = NGROK_TOKEN
!ngrok config add-authtoken "$NGROK_TOKEN"

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [4]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/main/projects/presentations/graph-med/")

DATA_PATH       = DRIVE_DIR / "ontology_graph.pkl"
GT_PATH         = DRIVE_DIR / "umls_ground_truth.csv"
CHECKPOINT_PATH = DRIVE_DIR / "gretriever_pcst_checkpoint.pt"
LORA_PATH       = DRIVE_DIR / "gretriever_pcst_lora"

assert DATA_PATH.exists(), f"Not found: {DATA_PATH}"
print(f"Drive mounted. Data dir: {DRIVE_DIR}")
print(f"  ontology_graph.pkl            : {'OK' if DATA_PATH.exists() else 'MISSING'}")
print(f"  umls_ground_truth.csv         : {'OK' if GT_PATH.exists() else 'MISSING'}")
print(f"  gretriever_pcst_checkpoint.pt : {'OK' if CHECKPOINT_PATH.exists() else 'MISSING'}")
print(f"  gretriever_pcst_lora/         : {'OK' if LORA_PATH.exists() else 'MISSING'}")

API_HOST = "0.0.0.0"
API_PORT = 8003

Mounted at /content/drive
Drive mounted. Data dir: /content/drive/MyDrive/main/projects/presentations/graph-med
  ontology_graph.pkl            : OK
  umls_ground_truth.csv         : OK
  gretriever_pcst_checkpoint.pt : OK
  gretriever_pcst_lora/         : OK


## HuggingFace Login

In [5]:
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("Logged in.")

Logged in.


## Model Definitions

In [6]:
class GRetrieverPCST(nn.Module):
    """
    GRetriever with PCST subgraph input and per-node soft tokens.

    Instead of mean-pooling all nodes into one vector, this model selects
    key nodes (ICD center + HPO candidates) and projects each independently
    to one soft token. This preserves per-node structural identity.
    """

    def __init__(
        self,
        in_channels=3584,
        gnn_hidden=256,
        gnn_out=256,
        num_layers=2,
        heads=4,
        dropout=0.1,
        llm_hidden_size=2048,
        max_key_nodes=16,
    ):
        super().__init__()
        self.llm_hidden_size = llm_hidden_size
        self.max_key_nodes = max_key_nodes

        self.input_proj = nn.Linear(in_channels, gnn_hidden)
        self.convs = nn.ModuleList()
        for i in range(num_layers):
            if i == num_layers - 1:
                self.convs.append(
                    GATConv(gnn_hidden, gnn_out, heads=1, concat=False, dropout=dropout)
                )
            else:
                self.convs.append(
                    GATConv(gnn_hidden, gnn_hidden // heads, heads=heads,
                            concat=True, dropout=dropout)
                )

        # MLP: per-node gnn_out -> one soft token (llm_hidden_size)
        self.node_proj = nn.Sequential(
            nn.Linear(gnn_out, llm_hidden_size),
            nn.GELU(),
            nn.Linear(llm_hidden_size, llm_hidden_size),
        )
        self.dropout = dropout

    def encode_nodes(self, x, edge_index):
        """GNN forward -> per-node embeddings [N, gnn_out]."""
        h = self.input_proj(x)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index)
            if i < len(self.convs) - 1:
                h = F.relu(h)
                h = F.dropout(h, p=self.dropout, training=self.training)
        return h  # [N, gnn_out]

    def forward(self, x, edge_index, key_node_ids):
        """
        Per-node forward: subgraph -> select key nodes -> soft tokens.

        Args:
            x:            [N, in_channels] node features
            edge_index:   [2, E] edges
            key_node_ids: list of local node indices to select (ICD + candidates)

        Returns:
            soft_tokens: [1, K, llm_hidden_size] where K = len(key_node_ids)
        """
        node_embs = self.encode_nodes(x, edge_index)  # [N, gnn_out]

        key_ids = key_node_ids[:self.max_key_nodes]
        key_ids_t = torch.tensor(key_ids, dtype=torch.long, device=x.device)
        key_embs = node_embs[key_ids_t]  # [K, gnn_out]

        soft_tokens = self.node_proj(key_embs)  # [K, llm_hidden_size]
        return soft_tokens.unsqueeze(0)  # [1, K, llm_hidden_size]


print("GRetrieverPCST class defined (per-node soft tokens).")

GRetrieverPCST class defined (per-node soft tokens).


## Load Graph + Models

In [7]:
import random
import pandas as pd
from torch_geometric.data import Data

# ── Load graph ──────────────────────────────────────────────────────────────
with open(DATA_PATH, "rb") as f:
    payload = pickle.load(f)

raw_x = torch.tensor(np.array(payload["x"]), dtype=torch.float32)
edge_index = torch.tensor(
    [payload["edge_src"], payload["edge_tgt"]], dtype=torch.long
)
data = Data(x=raw_x, edge_index=edge_index, num_nodes=raw_x.shape[0])
node_info = payload["node_info"]

code_to_idx = {n["code"]: i for i, n in enumerate(node_info) if n["code"]}
idx_to_name = {i: n.get("name", n["code"]) for i, n in enumerate(node_info)}

hpo_indices = [
    i for i, n in enumerate(node_info)
    if "HpoPhenotype" in n["labels"] and n["code"]
]
hpo_emb_norm = F.normalize(raw_x[hpo_indices], dim=1)

print(f"Graph: {data.num_nodes} nodes, {data.edge_index.shape[1]} edges")
print(f"HPO phenotypes: {len(hpo_indices)}")

# ── Augment with UMLS train edges (same 72/8/20 split as training) ──────────
gt_df = pd.read_csv(GT_PATH)
umls_edges = []
for _, row in gt_df.iterrows():
    icd_idx = code_to_idx.get(row["icd_id"])
    hpo_idx = code_to_idx.get(row["hpo_id"])
    if icd_idx is not None and hpo_idx is not None:
        umls_edges.append((icd_idx, hpo_idx))

random.seed(42)
random.shuffle(umls_edges)
n_train = int(0.72 * len(umls_edges))
train_umls = umls_edges[:n_train]

train_src = [e[0] for e in train_umls] + [e[1] for e in train_umls]
train_tgt = [e[1] for e in train_umls] + [e[0] for e in train_umls]
aug_edge_index = torch.cat([
    edge_index,
    torch.tensor([train_src, train_tgt], dtype=torch.long)
], dim=1)

data_augmented = Data(x=raw_x, edge_index=aug_edge_index, num_nodes=raw_x.shape[0])
print(f"Augmented graph: {data_augmented.edge_index.shape[1]} edges "
      f"(+{len(train_umls)*2} UMLS train, bidirectional)")

# ── Build adjacency list from augmented graph ───────────────────────────────
adj = defaultdict(set)
src_list, tgt_list = aug_edge_index[0].tolist(), aug_edge_index[1].tolist()
for s, t in zip(src_list, tgt_list):
    adj[s].add(t)
    adj[t].add(s)
print(f"Adjacency list: {len(adj)} nodes")


/tmp/ipykernel_2640/1400973275.py:10: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor(


Graph: 45205 nodes, 330488 edges
HPO phenotypes: 19944
Augmented graph: 333210 edges (+2722 UMLS train, bidirectional)
Adjacency list: 44667 nodes


In [8]:
from transformers import (
    AutoProcessor, AutoModelForImageTextToText,
    AutoTokenizer, BitsAndBytesConfig,
)
from peft import PeftModel

MODEL_ID = "google/medgemma-4b-it"
device = torch.device("cuda")

# ── Load LLM (4-bit) + LoRA adapter ─────────────────────────────────────────
print(f"Loading {MODEL_ID} (4-bit)...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

base_llm = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    quantization_config=quantization_config,
    device_map="auto",
)

# Fix chat template
if getattr(tokenizer, "chat_template", None) is None:
    ref_tok = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
    tokenizer.chat_template = ref_tok.chat_template
    tokenizer.bos_token = ref_tok.bos_token
    tokenizer.eos_token = ref_tok.eos_token

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

llm_hidden_size = base_llm.config.text_config.hidden_size
print(f"LLM hidden size: {llm_hidden_size}")

# Apply saved LoRA adapter
print(f"Loading LoRA adapter from {LORA_PATH}...")
llm = PeftModel.from_pretrained(base_llm, str(LORA_PATH))
llm.eval()

word_embedding = llm.get_input_embeddings()
print(f"LLM + LoRA loaded.")

Loading google/medgemma-4b-it (4-bit)...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

LLM hidden size: 2560
Loading LoRA adapter from /content/drive/MyDrive/main/projects/presentations/graph-med/gretriever_pcst_lora...
LLM + LoRA loaded.


In [9]:
# ── Load GNN checkpoint ─────────────────────────────────────────────────────
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
hp = checkpoint["hyperparams"]

MAX_KEY_NODES = hp.get("max_key_nodes", hp.get("num_tokens", 16))

gretriever = GRetrieverPCST(
    in_channels=hp["in_channels"],
    gnn_hidden=hp["gnn_hidden"],
    gnn_out=hp["gnn_out"],
    llm_hidden_size=llm_hidden_size,
    max_key_nodes=MAX_KEY_NODES,
).to(device).float()

gretriever.load_state_dict(checkpoint["gnn_state_dict"])
gretriever.eval()

print(f"GRetriever loaded: max {MAX_KEY_NODES} per-node soft tokens")
print(f"Training results: {checkpoint.get('results', {})}")

GRetriever loaded: max 16 per-node soft tokens
Training results: {'cosine_top1': 0.689419795221843, 'text_only': 0.7440273037542662, 'gretriever_pcst': 0.7406143344709898}


## PCST Subgraph + Inference Logic

In [10]:
SYSTEM_PROMPT = (
    "You are a medical ontology expert. Given an ICD-10 disease and a list of HPO "
    "phenotype candidates with graph context, select the best matching HPO phenotype. "
    "Output ONLY the HPO ID (e.g., HP:0000819). Nothing else."
)


def _verbalize_subgraph(node_ids, edges, node_info):
    """Convert subgraph to text description for LLM context."""
    lines = []
    for i, gid in enumerate(node_ids):
        info = node_info[gid]
        label = info.get("name", info.get("code", f"node_{gid}"))
        node_type = info["labels"][0] if info.get("labels") else "Unknown"
        lines.append(f"  [{i}] {node_type}: {label}")

    text = "Graph nodes:\n" + "\n".join(lines[:30])
    if len(lines) > 30:
        text += f"\n  ... and {len(lines) - 30} more nodes"

    if edges:
        edge_lines = []
        for s, t in edges:
            s_name = node_info[node_ids[s]].get("name", str(s))
            t_name = node_info[node_ids[t]].get("name", str(t))
            edge_lines.append(f"  [{s}] {s_name} -- [{t}] {t_name}")
        text += "\n\nGraph edges:\n" + "\n".join(edge_lines[:30])
        if len(edge_lines) > 30:
            text += f"\n  ... and {len(edge_lines) - 30} more edges"

    return text


def extract_pcst_subgraph(
    icd_idx, candidate_idxs, candidate_scores,
    num_hops=1, cost_e=0.5, max_nodes=64,
):
    """Extract PCST-pruned subgraph for an ICD->HPO query.

    Returns sub_x, sub_edge_index, sub_global_ids, text_desc, key_local_ids, viz_data.
    """
    seed_nodes = set([icd_idx] + candidate_idxs)

    # Expand neighborhood
    neighborhood = set(seed_nodes)
    frontier = set(seed_nodes)
    for _ in range(num_hops):
        next_frontier = set()
        for n in frontier:
            next_frontier.update(adj.get(n, set()))
        neighborhood.update(next_frontier)
        frontier = next_frontier - neighborhood

    neighborhood = sorted(neighborhood)
    if len(neighborhood) < 3:
        neighborhood = sorted(seed_nodes)

    global_to_local = {g: l for l, g in enumerate(neighborhood)}
    n_local = len(neighborhood)

    # Prizes
    prizes = np.zeros(n_local, dtype=np.float64)
    if icd_idx in global_to_local:
        prizes[global_to_local[icd_idx]] = 5.0
    for c_idx, score in zip(candidate_idxs, candidate_scores):
        if c_idx in global_to_local:
            prizes[global_to_local[c_idx]] = float(score) * 4.0
    for g_idx in neighborhood:
        l_idx = global_to_local[g_idx]
        if prizes[l_idx] == 0:
            prizes[l_idx] = 0.1

    # Local edges
    local_edges = []
    local_edge_costs = []
    seen_edges = set()
    for s, t in zip(src_list, tgt_list):
        if s in global_to_local and t in global_to_local:
            ls, lt = global_to_local[s], global_to_local[t]
            edge_key = (min(ls, lt), max(ls, lt))
            if edge_key not in seen_edges:
                seen_edges.add(edge_key)
                local_edges.append([ls, lt])
                local_edge_costs.append(cost_e)

    # Run PCST
    pcst_selected = set()
    if local_edges:
        edges_np = np.array(local_edges, dtype=np.int64)
        costs_np = np.array(local_edge_costs, dtype=np.float64)
        root = global_to_local.get(icd_idx, 0)
        selected_nodes, _ = pcst_fast.pcst_fast(
            edges_np, prizes, costs_np, root, 1, "strong", 0
        )
        pcst_selected = set(selected_nodes)

    # Force-include seed nodes
    seed_local = set()
    for g in seed_nodes:
        if g in global_to_local:
            seed_local.add(global_to_local[g])

    final_selected = pcst_selected | seed_local

    # Cap subgraph size
    if len(final_selected) > max_nodes:
        pcst_only = final_selected - seed_local
        pcst_by_prize = sorted(pcst_only, key=lambda n: -prizes[n])
        keep_pcst = set(pcst_by_prize[:max_nodes - len(seed_local)])
        final_selected = seed_local | keep_pcst

    # Map back to global indices
    sub_global_ids = sorted([neighborhood[l] for l in final_selected])
    sub_global_set = set(sub_global_ids)
    sub_g2l = {g: i for i, g in enumerate(sub_global_ids)}

    sub_edges_src, sub_edges_tgt = [], []
    for s, t in zip(src_list, tgt_list):
        if s in sub_global_set and t in sub_global_set:
            sub_edges_src.append(sub_g2l[s])
            sub_edges_tgt.append(sub_g2l[t])

    sub_x = data.x[sub_global_ids]
    if sub_edges_src:
        sub_edge_index = torch.tensor([sub_edges_src, sub_edges_tgt], dtype=torch.long)
    else:
        sub_edge_index = torch.zeros(2, 0, dtype=torch.long)

    text_desc = _verbalize_subgraph(sub_global_ids, list(zip(sub_edges_src, sub_edges_tgt)), node_info)

    # Key node local indices: ICD center + candidates
    key_local_ids = []
    if icd_idx in sub_g2l:
        key_local_ids.append(sub_g2l[icd_idx])
    for c_idx in candidate_idxs:
        if c_idx in sub_g2l:
            key_local_ids.append(sub_g2l[c_idx])

    # Build visualization data
    viz_nodes = []
    for i, gid in enumerate(sub_global_ids):
        info = node_info[gid]
        # Recover prize: map back to neighborhood local index
        local_in_neighborhood = global_to_local.get(gid, -1)
        prize = float(prizes[local_in_neighborhood]) if local_in_neighborhood >= 0 else 0.0
        node_type = info["labels"][0] if info.get("labels") else "Unknown"
        viz_nodes.append({
            "id": i,
            "name": info.get("name", info.get("code", f"node_{gid}")),
            "code": info.get("code", ""),
            "type": node_type,
            "prize": round(prize, 3),
        })

    # Deduplicate edges for viz (undirected)
    viz_edges = []
    seen_viz = set()
    for s, t in zip(sub_edges_src, sub_edges_tgt):
        edge_key = (min(s, t), max(s, t))
        if edge_key not in seen_viz:
            seen_viz.add(edge_key)
            viz_edges.append({"source": edge_key[0], "target": edge_key[1], "cost": cost_e})

    viz_data = {"nodes": viz_nodes, "edges": viz_edges}

    return sub_x, sub_edge_index, sub_global_ids, text_desc, key_local_ids, viz_data


def _build_prompt(icd_code, icd_name, candidates, text_desc):
    """Build text prompt from ICD code, candidate list, and graph context."""
    cand_lines = []
    for i, c in enumerate(candidates, 1):
        cand_lines.append(f"{i}. {c['code']} | {c['name']}")

    prompt = (
        f"ICD-10: {icd_code} | {icd_name}\n\n"
        f"HPO candidates:\n" + "\n".join(cand_lines) + "\n\n"
        f"Graph context:\n{text_desc}\n\n"
        f"Best matching HPO ID:"
    )
    return prompt


@torch.no_grad()
def _generate(icd_code, candidate_codes, use_graph_tokens=True):
    """
    Run GRetriever inference for a single ICD code.

    Returns:
        predicted HPO code, subgraph stats (including graph_context and viz_data)
    """
    icd_idx = code_to_idx.get(icd_code)
    if icd_idx is None:
        return None, {"error": f"ICD code '{icd_code}' not in graph"}

    icd_name = idx_to_name[icd_idx]

    # Resolve candidate indices
    valid_candidates = []
    for c in candidate_codes:
        cidx = code_to_idx.get(c)
        if cidx is not None:
            valid_candidates.append({
                "code": c,
                "idx": cidx,
                "name": idx_to_name[cidx],
            })

    if not valid_candidates:
        return None, {"error": "No valid candidates in graph"}

    # Cosine scores for PCST prizes
    icd_emb = F.normalize(data.x[icd_idx].unsqueeze(0), dim=1)
    cand_idxs = [c["idx"] for c in valid_candidates]
    cand_embs = F.normalize(data.x[cand_idxs], dim=1)
    cand_scores = (icd_emb @ cand_embs.T).squeeze(0).tolist()

    # PCST subgraph
    sub_x, sub_ei, sub_ids, text_desc, key_local_ids, viz_data = extract_pcst_subgraph(
        icd_idx, cand_idxs, cand_scores,
    )

    # Build prompt
    prompt = _build_prompt(icd_code, icd_name, valid_candidates, text_desc)

    # Tokenize
    messages = [{"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{prompt}"}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors="pt", padding=True)
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    text_embeds = word_embedding(input_ids)

    if use_graph_tokens:
        # GNN forward with per-node key selection
        soft_tokens = gretriever(
            sub_x.to(device), sub_ei.to(device), key_local_ids
        ).to(torch.bfloat16)
        n_graph_tokens = soft_tokens.shape[1]
        combined_embeds = torch.cat([soft_tokens, text_embeds], dim=1)
        graph_mask = torch.ones(1, n_graph_tokens, device=device)
        combined_mask = torch.cat([graph_mask, attention_mask], dim=1)
    else:
        combined_embeds = text_embeds
        combined_mask = attention_mask

    # Generate
    with torch.autocast("cuda", dtype=torch.bfloat16):
        outputs = llm.generate(
            inputs_embeds=combined_embeds,
            attention_mask=combined_mask,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    match = re.search(r"HP:\d{7}", generated)
    pred = match.group(0) if match else generated.strip()[:20]

    pred_name = idx_to_name.get(code_to_idx.get(pred, -1), "unknown")

    stats = {
        "subgraph_nodes": sub_x.shape[0],
        "subgraph_edges": sub_ei.shape[1],
        "num_candidates": len(valid_candidates),
        "num_key_nodes": len(key_local_ids),
        "graph_context": text_desc,
        "viz": viz_data,
    }

    return {"code": pred, "name": pred_name}, stats


print("Inference functions defined.")

Inference functions defined.


## FastAPI Application

In [11]:
# --- Pydantic models ---

class DisambiguateRequest(BaseModel):
    icd_code: str
    candidate_codes: List[str]


# --- App ---

app = FastAPI(title="GRetriever+PCST Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/healthz")
def healthz():
    return {
        "status": "ok",
        "model": "GRetrieverPCST",
        "llm": "medgemma-4b-it+LoRA",
        "max_key_nodes": MAX_KEY_NODES,
        "nodes": data.num_nodes,
        "edges": int(data.edge_index.shape[1]),
    }


@app.post("/disambiguate")
def disambiguate(req: DisambiguateRequest):
    """
    Run GRetriever (GNN soft tokens + LoRA MedGemma) on ICD + candidates.

    Returns the predicted HPO code with subgraph stats.
    """
    pred, stats = _generate(req.icd_code, req.candidate_codes, use_graph_tokens=True)
    if pred is None:
        return JSONResponse({"error": stats.get("error", "unknown")}, status_code=400)

    return JSONResponse({
        "icd_code": req.icd_code,
        "icd_label": idx_to_name.get(code_to_idx.get(req.icd_code, -1), req.icd_code),
        "prediction": pred,
        "method": "gretriever_pcst_lora",
        "subgraph": stats,
    })


@app.post("/compare")
def compare(req: DisambiguateRequest):
    """
    Run both text-only (LoRA, no graph tokens) and GRetriever (LoRA + graph tokens).

    Returns side-by-side predictions for easy comparison in the demo notebook.
    """
    gret_pred, gret_stats = _generate(
        req.icd_code, req.candidate_codes, use_graph_tokens=True
    )
    text_pred, text_stats = _generate(
        req.icd_code, req.candidate_codes, use_graph_tokens=False
    )

    if gret_pred is None:
        return JSONResponse({"error": gret_stats.get("error", "unknown")}, status_code=400)

    return JSONResponse({
        "icd_code": req.icd_code,
        "icd_label": idx_to_name.get(code_to_idx.get(req.icd_code, -1), req.icd_code),
        "text_only": text_pred,
        "gretriever": gret_pred,
        "subgraph": gret_stats,
    })


print("FastAPI app defined with endpoints: /healthz, /disambiguate, /compare")

FastAPI app defined with endpoints: /healthz, /disambiguate, /compare


## Serve with ngrok

In [ ]:
public_url = ngrok.connect(API_PORT, "http", domain=NGROK_DOMAIN_GRETRIEVER).public_url
print(f"Public endpoint: {public_url}")

config = uvicorn.Config(app, host=API_HOST, port=API_PORT, log_level="info")
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [2640]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8003 (Press CTRL+C to quit)


Public endpoint: https://gretriever-nodes2026.ngrok.io
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "GET /healthz HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "GET /healthz HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /compare HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /disambiguate HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /disambiguate HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /disambiguate HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /disambiguate HTTP/1.1" 200 OK
INFO:     2a01:e11:5401:8500:95f8:8617:8047:b5b4:0 - "POST /disambiguate HTTP